# tSCS EMG — the four comparison figures (subject NTA, 24-07-2026)

One session, one participant: **stimulation mode** (30 Hz burst / ARC-EX) × **polarity**
(cathodic / anodic) × **lidocaine** (before / with). Four figures, each holding one thing fixed:

| figure | fixed | compared | with |
|---|---|---|---|
| **1** | anodic | 30 Hz vs ARC-EX | before and with lidocaine |
| **2** | cathodic | 30 Hz vs ARC-EX | before and with lidocaine |
| **3** | ARC-EX | anodic vs cathodic | before and with lidocaine |
| **4** | 30 Hz burst | anodic vs cathodic | before and with lidocaine |

## Colours and styles
**gray = before lidocaine, orange = with lidocaine** in every figure. The second factor is the
**style**: in figures 1–2 **solid / plain bars = 30 Hz burst, dashed / hatched = ARC-EX**; in
figures 3–4 **solid / plain = cathodic, dashed / hatched = anodic**.

Burst is shown at `BURST_MA`, ARC-EX at `ARCEX_MA` — different mA, roughly matched relative to
their motor thresholds (burst 25–30, ARC-EX 65–90 mA from the log). Figures 3 and 4 compare one
mode with itself, so all four trains there are at the same intensity.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from functions import set_style, load_run, pretty
from functions.burst import compare_at_intensity, summary_curves, plot_pulse_overlay
from functions.paper import fig_train_modes
set_style()


## Config

In [ ]:
D = "tSCS_CHUV_data/24-07-2026/testSCS/"
BURST_MA, ARCEX_MA = 35, 110          # <-- intensity of each mode

FILE = {   # (mode, polarity, lidocaine state) -> file
    ("burst", "cathodic", "before"):    "Burst_autosave_20260724_102443_670ms.csv",
    ("burst", "cathodic", "lidocaine"): "Burst_autosave_20260724_113834_522ms.csv",
    ("burst", "anodic",   "before"):    "Burst_autosave_20260724_102721_694ms.csv",
    ("burst", "anodic",   "lidocaine"): "Burst_autosave_20260724_114055_891ms.csv",
    ("arcex", "cathodic", "before"):    "Modulated_autosave_20260724_103530_508ms.csv",
    ("arcex", "cathodic", "lidocaine"): "Modulated_autosave_20260724_114332_473ms.csv",
    ("arcex", "anodic",   "before"):    "Modulated_autosave_20260724_103849_459ms.csv",
    ("arcex", "anodic",   "lidocaine"): "Modulated_autosave_20260724_114730_561ms.csv",
}
NAME = {"burst": "30 Hz", "arcex": "ARC-EX", "cathodic": "cathodic", "anodic": "anodic"}
LIDO_COL = {"before": "0.45", "lidocaine": "#f39c12"}       # colour = lidocaine
STYLE_A  = ("",    "-")                                      # first compared value: plain / solid
STYLE_B  = ("///", "--")                                     # second: hatched / dashed
AMP      = {"burst": BURST_MA, "arcex": ARCEX_MA}

N_PULSES, RESP_START_MS, GUARD_MS, MIN_SNR, MAX_EDGE_FRAC = 10, 8.0, 1.0, 2.0, 0.5
KW = dict(n_pulses=N_PULSES, resp_start_ms=RESP_START_MS, guard_ms=GUARD_MS, min_snr=MIN_SNR,
          max_edge_frac=MAX_EDGE_FRAC)

muscles = [c for c in load_run(D + FILE[("burst", "cathodic", "before")])[2] if c != "Trigger A"]

def build(fixed, compare):
    """fixed: ('polarity', 'anodic') or ('mode', 'burst'); compare: the two values of the other
    factor, e.g. ('burst', 'arcex') or ('cathodic', 'anodic'). Returns everything the plotting
    functions need, ordered: A-before, A-lidocaine, B-before, B-lidocaine."""
    files, labels, colours, hatches, lss, amps = [], [], [], [], [], []
    for val, (hat, ls) in zip(compare, (STYLE_A, STYLE_B)):
        for state in ("before", "lidocaine"):
            key = (val, fixed[1], state) if fixed[0] == "polarity" else (fixed[1], val, state)
            files.append(D + FILE[key])
            labels.append(f"{NAME[val]} · {state}")
            colours.append(LIDO_COL[state]); hatches.append(hat); lss.append(ls)
            amps.append(AMP[key[0]])
    return dict(files=files, labels=labels, colours=colours, hatches=hatches,
                linestyles=lss, amps=tuple(amps))

def show(cfg, title, muscle=None):
    compare_at_intensity(cfg["files"], None, amp=cfg["amps"], normalize="none", muscles=muscle,
                         labels=cfg["labels"], colours=cfg["colours"], hatches=cfg["hatches"],
                         linestyles=cfg["linestyles"], title=title, **KW)

for k, f in FILE.items():
    meta, _, _ = load_run(D + f)
    a = AMP[k[0]]
    print(f"{NAME[k[0]]:7s} {k[1]:9s} {k[2]:10s} {a:4d} mA " + ("ok" if a in [m['amp_ma'] for m in meta] else "MISSING"))


## Figure 1 · 30 Hz vs ARC-EX — **anodic**, before and with lidocaine

Solid / plain bars = 30 Hz burst, dashed / hatched = ARC-EX; gray = before, orange = with lidocaine.

In [ ]:
FIG1 = build(("polarity", "anodic"), ("burst", "arcex"))
show(FIG1, "Fig 1 · anodic — 30 Hz vs ARC-EX")


### Figure 1 · one muscle

In [ ]:
MUSCLE = "Flex. digitorum (R)"          # label from any panel title, or a channel name
show(FIG1, f"Fig 1 · anodic — {MUSCLE}", muscle=MUSCLE)


## Figure 2 · 30 Hz vs ARC-EX — **cathodic**, before and with lidocaine

In [ ]:
FIG2 = build(("polarity", "cathodic"), ("burst", "arcex"))
show(FIG2, "Fig 2 · cathodic — 30 Hz vs ARC-EX")


### Figure 2 · one muscle

In [ ]:
show(FIG2, f"Fig 2 · cathodic — {MUSCLE}", muscle=MUSCLE)


## Figure 3 · ARC-EX — **anodic vs cathodic**, before and with lidocaine

Solid / plain = cathodic, dashed / hatched = anodic. All four trains at `ARCEX_MA`.

In [ ]:
FIG3 = build(("mode", "arcex"), ("cathodic", "anodic"))
show(FIG3, "Fig 3 · ARC-EX — cathodic vs anodic")


### Figure 3 · one muscle, and across intensities

In [ ]:
show(FIG3, f"Fig 3 · ARC-EX — {MUSCLE}", muscle=MUSCLE)
summary_curves(FIG3["files"], None, labels=FIG3["labels"], colours=FIG3["colours"],
               markers=["o", "o", "s", "s"], **KW);


## Figure 4 · 30 Hz burst — **anodic vs cathodic**, before and with lidocaine

All four trains at `BURST_MA`.

In [ ]:
FIG4 = build(("mode", "burst"), ("cathodic", "anodic"))
show(FIG4, "Fig 4 · 30 Hz burst — cathodic vs anodic")


### Figure 4 · one muscle, and across intensities

In [ ]:
show(FIG4, f"Fig 4 · 30 Hz burst — {MUSCLE}", muscle=MUSCLE)
summary_curves(FIG4["files"], None, labels=FIG4["labels"], colours=FIG4["colours"],
               markers=["o", "o", "s", "s"], **KW);


## Paper-style version — traces + pulse 1 vs rest, 2–3 muscles

`fig_train_modes` for any pair of conditions; set `SAVE` to write a 300 dpi PNG.

In [ ]:
MUSCLES_FIG = ["Flex. digitorum (R)", "Flex. carpi rad. (R)", "Ext. digitorum (L)"]
SAVE = None        # e.g. "figures/fig1_anodic_burst_vs_arcex.png"

specs = [dict(label="30 Hz burst", csv=D + FILE[("burst", "anodic", "before")], amp=BURST_MA, colour="black"),
         dict(label="ARC-EX",      csv=D + FILE[("arcex", "anodic", "before")], amp=ARCEX_MA, colour="#d62728")]
fig_train_modes(specs, MUSCLES_FIG, n_pulses=N_PULSES, resp_start_ms=RESP_START_MS,
                title="anodic · before lidocaine", save=SAVE);
